# Buổi 23 · Capstone: ảnh hoá đơn vào, sổ chi tiêu ra

Web app FastAPI ráp cả khoá: **SigLIP 2** (ViT) kiểm ảnh, **PaddleOCR** tìm khung và đọc chữ, sensor cộng tiền. Đọc không chắc thì **Textract** và **Nova Lite** đọc lại, kiểm bằng sensor độc lập. Vẫn không chắc thì chuyển người.

Bạn làm 5 TODO trong `logic.py`, phần logic lõi của harness. Mọi file khác đã có sẵn.

Cách chính là chạy trên máy mình (xem `README.md`, phần "Chạy trên máy"). Notebook này dành cho máy yếu hoặc không cài được Python 3.12.

Colab xoá file khi tắt phiên: làm xong nhớ tải `logic.py` về máy (chuột phải → Download).

## 1. Cài thư viện (nên chạy trước giờ học)

Khoảng 3–5 phút. Cài xong mà Colab hiện nút **Restart session** thì bấm vào, rồi chạy tiếp từ ô sau, bỏ qua ô cài này.

In [ ]:
!git clone -q -b dev https://github.com/lequocduyquang/receipt-capstone.git /content/receipt-capstone
%cd /content/receipt-capstone
!pip install -q -r requirements-dev.txt

## 2. Làm 5 TODO trong `logic.py`

Mở `logic.py`: biểu tượng thư mục bên trái, vào `receipt-capstone/`, bấm đúp. Sửa xong bấm **Ctrl + S**.

| TODO | Hàm | Việc cần làm | Kiểm bằng |
|---|---|---|---|
| 1 | `quyet_cong_vao` | Kiểm mờ trước, rồi SigLIP chắc thì cho qua hoặc chặn, lưng chừng thì hỏi Nova | `pytest -k todo1` |
| 2 | `kiem_tra` | Các món cộng lại = tạm tính; tạm tính − giảm giá = tổng | `pytest -k todo2` |
| 3 | `can_fallback` | Sensor lỗi hoặc điểm đọc thấp thì gọi Bedrock | `pytest -k todo3` |
| 4 | `kiem_nova` | Tổng Nova khai phải nằm trên dòng TỔNG CỘNG mà Textract đọc được | `pytest -k todo4` |
| 5 | `cong_duyet` | Chỉ ghi sổ khi có bằng chứng; hai nguồn đọc tổng khác nhau thì chuyển người | `pytest -k todo5` |

Chưa làm thì test báo `NotImplementedError`. Xong cả 5 thì `pytest` báo **21 passed**. Test không tải model, không cần key AWS, chạy dưới 3 giây.

In [ ]:
%cd /content/receipt-capstone   # chạy lại ô này sau mỗi lần Restart session

In [ ]:
!python -m pytest -q -k todo1

In [ ]:
!python -m pytest -q

## 3. Chạy web app

Ô dưới chạy server FastAPI ở nền, lần đầu tải khoảng 1,5 GB model (2–3 phút), rồi mở trang web ngay trong Colab. Thử 6 ảnh mẫu:

| Ảnh mẫu | Kết quả đúng |
|---|---|
| `easy_01`, `medium_02`, `hard_06` | `GHI SỔ`, nguồn `paddle` |
| `che_tong` | `CẦN NGƯỜI XEM`: không thấy dòng TỔNG CỘNG |
| `hoa_don_mo` | `CHẶN`: ảnh bị mờ |
| `khong_phai_hoa_don` | `CHẶN`: không giống hoá đơn |

Colab không có key AWS nên bước 4 (fallback) báo "cần đọc lại nhưng máy không có key AWS". Đó là đúng thiết kế: không có quyền gọi AWS thì chuyển người, không đoán.

In [ ]:
import subprocess, time, urllib.request

server = subprocess.Popen(["python", "-m", "uvicorn", "server:app", "--port", "7860"])
for _ in range(300):   # chờ tải model xong
    try:
        urllib.request.urlopen("http://127.0.0.1:7860/healthz")
        break
    except Exception:
        time.sleep(1)

from google.colab import output
output.serve_kernel_port_as_window(7860)

Sửa `logic.py` xong muốn chạy lại thì tắt server cũ trước: `server.terminate()`, rồi chạy lại ô trên.

## 4. Eval: harness có để lọt phiếu sai không?

Chạy 10 hoá đơn ở 3 dạng: ảnh gốc, ảnh bị che dòng TỔNG CỘNG (bài kiểm của Buổi 22, Nova + harness lọt 8/10), và ảnh chất lượng thấp. Cột quan trọng nhất là **sai mà lọt**.

In [ ]:
!python danhgia.py --n 10

Colab có 2 CPU. Chạy song song 2 bản bằng Ray, mỗi bản tải model một lần, rồi so số giây với ô trên:

In [ ]:
!python danhgia.py --n 10 --ray 2